In [1]:
!pip install tensorflow opencv-python scikit-learn matplotlib gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
from sklearn.model_selection import train_test_split


In [8]:
import os
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import gradio as gr
from sklearn.metrics import accuracy_score
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, DenseNet121, MobileNetV2, InceptionV3, InceptionResNetV2


In [11]:
IMG_SIZE = 128
def load_data(image_dir, mask_dir):
    images, masks = [], []
    image_files = sorted(os.listdir(image_dir))
    mask_files = sorted(os.listdir(mask_dir))

    for img_file, mask_file in zip(image_files, mask_files):
        img_path = os.path.join(image_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)
        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if img is None or mask is None:
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
        mask = np.expand_dims((mask > 0).astype(np.uint8), axis=-1)

        images.append(img)
        masks.append(mask)

    return np.array(images), np.array(masks)

# Load Dataset 1 (for training)
X_train, y_train = load_data('/content/drive/MyDrive/COVID/frames_train/', '/content/drive/MyDrive/COVID/masks_train/')

# Load Dataset 2 (for testing)
X_test, y_test = load_data('/content/drive/MyDrive/COVID2/frames_test/', '/content/drive/MyDrive/COVID2/masks_test/')


In [13]:
def build_unet_resnet_hybrid(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    inputs = tf.keras.Input(shape=input_shape)
    base_model = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_tensor=inputs)
    base_model.trainable = False

    skips = [
        base_model.get_layer("conv1_relu").output,
        base_model.get_layer("conv2_block3_out").output,
        base_model.get_layer("conv3_block4_out").output,
        base_model.get_layer("conv4_block6_out").output
    ]
    x = base_model.get_layer("conv5_block3_out").output

    for skip in reversed(skips):
        x = tf.keras.layers.UpSampling2D((2, 2))(x)
        x = tf.keras.layers.Concatenate()([x, skip])
        x = tf.keras.layers.Conv2D(256, 3, activation='relu', padding='same')(x)
        x = tf.keras.layers.Conv2D(256, 3, activation='relu', padding='same')(x)

    x = tf.keras.layers.UpSampling2D((2, 2))(x)
    x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)

    outputs = tf.keras.layers.Conv2D(1, 1, activation='sigmoid')(x)
    return tf.keras.Model(inputs, outputs)


In [14]:
model = build_unet_resnet_hybrid()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=6, batch_size=8, validation_split=0.1, steps_per_epoch=100)
model.save('/content/drive/MyDrive/COVID_Models/unet_resnet_model.h5')


Epoch 1/6
100/100 ━━━━━━━━━━━━━━━━━━━━ 64s 317ms/step - accuracy: 0.9360 - loss: 0.2295 - val_accuracy: 0.9656 - val_loss: 0.1920
Epoch 2/6
100/100 ━━━━━━━━━━━━━━━━━━━━ 16s 157ms/step - accuracy: 0.9843 - loss: 0.0645 - val_accuracy: 0.9656 - val_loss: 0.1664
Epoch 3/6
100/100 ━━━━━━━━━━━━━━━━━━━━ 16s 160ms/step - accuracy: 0.9833 - loss: 0.0596 - val_accuracy: 0.9656 - val_loss: 0.3690
Epoch 4/6
  7/100 ━━━━━━━━━━━━━━━━━━━━ 13s 144ms/step - accuracy: 0.9864 - loss: 0.0536

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:107: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.9885 - loss: 0.0474 - val_accuracy: 0.9656 - val_loss: 0.1759
Epoch 5/6
100/100 ━━━━━━━━━━━━━━━━━━━━ 16s 163ms/step - accuracy: 0.9843 - loss: 0.0579 - val_accuracy: 0.9656 - val_loss: 0.1692
Epoch 6/6
100/100 ━━━━━━━━━━━━━━━━━━━━ 16s 162ms/step - accuracy: 0.9852 - loss: 0.0546 - val_accuracy: 0.9656 - val_loss: 0.1461


In [15]:
def evaluate_model(model, X, y):
    preds = model.predict(X)
    preds = (preds > 0.5).astype(np.uint8)
    acc = accuracy_score(y.flatten(), preds.flatten())
    return acc

unet_resnet_model = tf.keras.models.load_model('/content/drive/MyDrive/COVID_Models/unet_resnet_model.h5')
test_acc = evaluate_model(unet_resnet_model, X_test, y_test)
print(f"✅ Test Accuracy on Dataset 2: {test_acc*100:.2f}%")


86/86 ━━━━━━━━━━━━━━━━━━━━ 48s 334ms/step
✅ Test Accuracy on Dataset 2: 98.34%


In [16]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

In [17]:
preds = (unet_resnet_model.predict(X_test) > 0.5).astype(np.uint8)
dice = dice_coef(y_test, preds)
print(f"🧪 Dice Coefficient on Dataset 2: {dice:.4f}")

86/86 ━━━━━━━━━━━━━━━━━━━━ 17s 192ms/step
🧪 Dice Coefficient on Dataset 2: 0.0000


In [19]:
import gradio as gr
import cv2


In [20]:
# Load trained model
model = tf.keras.models.load_model('/content/drive/MyDrive/COVID_Models/unet_resnet_model.h5')

# Prediction function
def predict_ct(image):
    # Resize to match model input
    image_resized = cv2.resize(image, (IMG_SIZE, IMG_SIZE)) / 255.0
    image_input = np.expand_dims(image_resized, axis=0)  # Add batch dimension

In [24]:
def predict_ct(image):
    image_resized = cv2.resize(image, (IMG_SIZE, IMG_SIZE)) / 255.0
    image_input = np.expand_dims(image_resized, axis=0)

    # Predict the mask
    pred_mask = model.predict(image_input)[0]
    pred_mask = (pred_mask > 0.5).astype(np.uint8) * 255

    # Create overlay
    overlay = image_resized.copy()
    overlay_mask = pred_mask.squeeze().astype(np.uint8)
    overlay_mask_colored = cv2.merge([overlay_mask]*3)
    overlay = (0.6 * overlay * 255 + 0.4 * overlay_mask_colored).astype(np.uint8)

    return overlay

In [29]:
def predict_ct(image):
    # Resize and normalize
    image_resized = cv2.resize(image, (IMG_SIZE, IMG_SIZE)) / 255.0
    image_input = np.expand_dims(image_resized, axis=0)

    # Predict the mask
    pred_mask = model.predict(image_input)[0]
    pred_mask_bin = (pred_mask > 0.5).astype(np.uint8) * 255
    pred_mask_bin = pred_mask_bin.squeeze()

    # Convert grayscale mask to 3 channels
    pred_mask_colored = cv2.merge([pred_mask_bin] * 3)

    # Convert original to display range
    image_disp = (image_resized * 255).astype(np.uint8)

    # Overlay mask on original image
    overlay = cv2.addWeighted(image_disp, 0.7, pred_mask_colored, 0.3, 0)

    # Stack all three: original | mask | overlay
    combined = np.hstack([image_disp, pred_mask_colored, overlay])
    return combined


In [30]:
gr.Interface(
    fn=predict_ct,
    inputs=gr.Image(type="numpy", label="Upload CT Image"),
    outputs=gr.Image(type="numpy", label="Original | Mask | Overlay"),
    title="COVID-19 CT Scan Segmentation",
    description="Left: Original CT, Middle: Infection Mask, Right: Overlay"
).launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b9834ba947a84e1a34.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
